# 20 · Independent review and query-level uncertainty

This is a **software exercise on fictional OSM data with deliberately simulated relevance labels**. It does not evaluate real restaurants. The real Boston study requires independent human reviewers for all 1,320 eligible pairs and a separately frozen analysis plan. This notebook checks reviewer progress, schema integrity, and the paired query-bootstrap report without model downloads or internet access.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from savoroute.data.catalog import catalog_to_dict
from savoroute.evaluation.annotations import merge_reviewer_sheets
from savoroute.evaluation.research import (
    audit_reviewer_progress, prepare_analysis_plan, analyze_reviewed_study,
)
from savoroute.evaluation.study import prepare_osm_study, verify_osm_study, StudyError
from savoroute.ingestion.osm import import_osm_snapshot

root = Path.cwd()
source = root / "data/sample/osm_overpass_fixture.json"
snapshot = json.loads(source.read_text(encoding="utf-8"))
imported = import_osm_snapshot(snapshot, synthetic_fixture=True)
temporary = TemporaryDirectory()
work = Path(temporary.name)
catalog_file = work / "catalog.json"
catalog_file.write_text(json.dumps(catalog_to_dict(imported.catalog)), encoding="utf-8")
print("Fictional catalog items:", len(imported.catalog.items))

In [ ]:
query_set = {"query_set_version": 1, "queries": [
    {"case_id": "dev-italian", "query": "Italian food", "filters": {}, "split": "development"},
    {"case_id": "dev-no-match", "query": "Volcanic dessert", "filters": {}, "split": "development"},
    {"case_id": "test-mediterranean", "query": "Mediterranean mezze", "filters": {}, "split": "test"},
    {"case_id": "test-no-match", "query": "Space buffet", "filters": {}, "split": "test"},
]}
query_file = work / "queries.json"
query_file.write_text(json.dumps(query_set), encoding="utf-8")
study_dir = work / "frozen"
manifest = prepare_osm_study(
    source_path=source, catalog_path=catalog_file, queries_path=query_file,
    output_dir=study_dir, collected_at="2026-11-12T12:00:00Z",
    selection_protocol="Fictional importer fixture only",
    query_protocol="Four manually written fictional examples",
    allow_synthetic_fixture=True,
)
assert verify_osm_study(study_dir) == manifest
print("Unlabeled pairs:", manifest["judgment_pairs"])
plan_path = work / "analysis-plan.json"
plan = prepare_analysis_plan(study_dir, plan_path, resamples=250, seed=2026)
print("Frozen primary metric:", plan["primary_metric"], "@", plan["at_k"])

In [ ]:
# Reviewer progress: the original frozen sheets are entirely blank.
blank_a = study_dir / "reviewer_a.json"
blank_b = study_dir / "reviewer_b.json"
audit_blank = audit_reviewer_progress(study_dir, blank_a, blank_b)
assert audit_blank["overall"]["both_complete"] == 0
print("Before independent review:", audit_blank["overall"])
print("Review UI: serve the app locally, then open /study-review and select the two files.")

## Simulated judgments (fixture only)

The next cell programmatically fills both reviewer sheets to test the software. This step is **not permitted for the real OSM study**: two different people must instead review their own blinded copies independently, with evidence-based rationales.

In [ ]:
catalog_item = imported.catalog.items[0].item_id
for original, name in ((blank_a, "a"), (blank_b, "b")):
    sheet = json.loads(original.read_text(encoding="utf-8"))
    for row in sheet["judgments"]:
        positive = (row["case_id"] in {"dev-italian", "test-mediterranean"}
                    and row["item_id"] == catalog_item)
        row["relevant"] = positive
        row["note"] = "Artificial software-test relevance" if positive else ""
    (work / f"reviewer_{name}.json").write_text(json.dumps(sheet), encoding="utf-8")
reviewer_a = work / "reviewer_a.json"
reviewer_b = work / "reviewer_b.json"
audit_complete = audit_reviewer_progress(study_dir, reviewer_a, reviewer_b)
assert audit_complete["overall"]["ready_to_merge"]
print("After simulated review:", audit_complete["overall"])

In [ ]:
template = json.loads((study_dir / "review_template.json").read_text(encoding="utf-8"))
merged = merge_reviewer_sheets(
    imported.catalog, template,
    json.loads(reviewer_a.read_text(encoding="utf-8")),
    json.loads(reviewer_b.read_text(encoding="utf-8")),
)
# All reviewers agree in this artificial case; no adjudication is needed.
workbook = work / "adjudicated.json"
workbook.write_text(json.dumps(merged), encoding="utf-8")
report = analyze_reviewed_study(study_dir, workbook, plan_path, split="development")
assert report["study_report"]["annotation_audit"]["synthetic"] is True
print("Development-only artificial comparison:")
for metric in ("ndcg_at_k", "recall_at_k", "correct_empty"):
    info = report["metrics"][metric]
    print(metric, "n_queries", info["n_queries"],
          "BM25 - overlap", info["bm25_minus_overlap"])
print("Error categories:", report["failure_counts"])

In [ ]:
try:
    analyze_reviewed_study(study_dir, workbook, plan_path, split="test")
except StudyError as exc:
    assert "allow_test" in str(exc)
    print("Held-out test gated:", exc)
else:
    raise AssertionError("test access unexpectedly permitted")

# Do not evaluate the held-out partition in this tutorial.
temporary.cleanup()

## Interpretation

A bootstrap interval with fewer than two queries in its positive or no-match stratum is reported as `null`. Even when two are present, a short hand-written query sample does **not** support statistical superiority or generalization claims. The confidence interval does not incorporate label disagreement, incompleteness of OSM fields, or changes in the external business world. For the full human-study protocol, see `docs/statistical-relevance-study.md`.